# Project Pipeline

One growing integration checkpoint. Loads the project dataset **once**, then runs each
stage's `src/` helper against it, top to bottom. The full analysis and write-up for each
stage lives in the matching homework notebook.

| stage | helper | write-up |
|---|---|---|
| 06 cleaning | `src/cleaning.py` | `homework/homework06/` |
| 07 outliers | `src/outliers.py` | `notebooks/sensitivity_outliers.ipynb` · `docs/outliers.md` |
| 08 EDA | `src/eda.py` | `notebooks/eda.ipynb` |
| 09 features | `src/features.py` | `homework/homework09/` |
| 10a regression | `src/features.py` | `notebooks/modeling-linear-regression.ipynb` |
| 10b classification | `src/features.py::add_ladder_features` | `notebooks/modeling-time-series-and-classification.ipynb` |
| 11 evaluation | `src/evaluation.py` | `homework/homework11/` · `data/processed/scenario_results.csv` |
| 12 reporting | `src/ev.py::ev_table` | `reports/stakeholder_report.md` (+ `.pdf`, `reports/images/`) · `homework/homework12/` |
| 13 productization | `src/model.py` · `src/plotting.py` · `app.py` | `README.md` · `homework/homework13/` |

Stage 13 extracts the Stage 10a model into `src/model.py`, re-runs it here below the
original cell to check the refactor changed nothing, then saves `model/model.pkl` and
serves it from `app.py`.

In [11]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# notebook lives in notebooks/; src/ and data/ are one level up
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))

from src import cleaning, outliers, features, evaluation, ev
from src.eda import eda_summary

pd.set_option("display.max_columns", 100)

## Load data

The project dataset: `data/raw/card_tiers.csv` (one row per hobby-box parallel / insert /
autograph tier) joined to a few box-level fields from `data/raw/box_products.csv`. Built by
`src/build_raw_dataset.py`; schema in `docs/data_dictionary.md`.

In [12]:
ct = pd.read_csv(project_root / "data" / "raw" / "card_tiers.csv")
bp = pd.read_csv(project_root / "data" / "raw" / "box_products.csv")
df = ct.merge(
    bp[["product_id", "product_line", "packs_per_box", "retail_price_usd", "release_date"]],
    on="product_id", how="left",
)
print(df.shape)
df.head()

(194, 17)


,product_id,box_format,tier_name,tier_group,print_run,is_numbered,is_autograph,is_ssp,odds_pack,odds_box,est_value_usd,value_basis,source_note,product_line,packs_per_box,retail_price_usd,release_date
0,2025-26-topps-chrome-bkb-hobby,Hobby,Base,base,NaN,False,False,False,NaN,NaN,0.5,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
1,2025-26-topps-chrome-bkb-hobby,Hobby,Base Refractor,parallel,NaN,False,False,False,3.0,0.15,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
2,2025-26-topps-chrome-bkb-hobby,Hobby,Prism Refractor,parallel,NaN,False,False,False,5.0,0.25,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
3,2025-26-topps-chrome-bkb-hobby,Hobby,Wave Refractor,parallel,NaN,False,False,False,14.0,0.70,3.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18
4,2025-26-topps-chrome-bkb-hobby,Hobby,Negative Refractor,parallel,NaN,False,False,False,31.0,1.55,8.0,rough_estimate_v0,NaN,2025-26 Topps Chrome Basketball,20,379.99,2025-12-18


## Stage 06 — cleaning  →  `src/cleaning.py`

`fill_missing_median` / `drop_missing` / `normalize_data` are available, but this dataset
needs **no row-level imputation** — the NaNs are structural, not dirty:

- `print_run` (~51% NaN) = the card is **unnumbered**, not missing. `is_numbered` encodes it; filling would invent print runs.
- `odds_pack` (~7% NaN) = Topps published **card-level** odds only for that tier, no aggregate. Left as NaN, handled per stage.

In [13]:
missing = df.isna().sum()
print(missing[missing > 0])
# Decision: keep the structural NaNs (see markdown above) -- no fill_missing_median /
# drop_missing here. cleaning.normalize_data is a modelling-time step, deferred to Stage 10.

print_run       99
odds_pack       13
odds_box        13
source_note    174
dtype: int64


## Stage 07 — outliers  →  `notebooks/sensitivity_outliers.ipynb` · `docs/outliers.md`

`src/outliers.py`: flag the rare "chase" tiers on the rarity axis (`odds_pack`). **Flag,
never remove** — those tiers carry most of a box's EV (`docs/outliers.md`).

In [14]:
mask_iqr = outliers.detect_outliers_iqr(df["odds_pack"])
df = outliers.flag_outliers(df, "odds_pack", mask_iqr, flag_column="is_chase")
print(f"chase tiers flagged: {int(mask_iqr.sum())} of {len(df)} ({mask_iqr.mean():.1%})  "
      f"-- flagged, not removed")
df.loc[df["is_chase"], ["product_line", "tier_name", "odds_pack", "est_value_usd"]].head(10)

chase tiers flagged: 29 of 194 (14.9%)  -- flagged, not removed


,product_line,tier_name,odds_pack,est_value_usd
21,2025-26 Topps Chrome Basketball,Red Refractor,4353.0,550.0
22,2025-26 Topps Chrome Basketball,Red Wave Refractor,3420.0,550.0
23,2025-26 Topps Chrome Basketball,FrozenFractor,4353.0,550.0
24,2025-26 Topps Chrome Basketball,SuperFractor,21767.0,4000.0
58,2025-26 Topps Chrome Update Series Basketball,Black Refractor,3022.0,300.0
60,2025-26 Topps Chrome Update Series Basketball,Red Refractor,6054.0,550.0
61,2025-26 Topps Chrome Update Series Basketball,Red Wave Refractor,3119.0,550.0
62,2025-26 Topps Chrome Update Series Basketball,FrozenFractor,6054.0,550.0
63,2025-26 Topps Chrome Update Series Basketball,SuperFractor,30361.0,4000.0
66,2025-26 Topps Chrome Update Series Basketball,Denim Tears,6054.0,150.0


## Stage 08 — EDA  →  `notebooks/eda.ipynb`

`src/eda.py`'s `eda_summary()` — quick profile of the project dataset.

In [15]:
summary = eda_summary(df)
print("shape:", summary["shape"])
print("missing:", {k: v for k, v in summary["missing"].items() if v})
summary["numeric_profile"]

shape: (194, 18)
missing: {'print_run': 99, 'odds_pack': 13, 'odds_box': 13, 'source_note': 174}


,count,mean,std,min,25%,50%,75%,max,skew,kurtosis
print_run,95.0,96.105263,106.471300,1.00,10.00,50.00,150.00,499.00,1.362879,1.640604
odds_pack,181.0,3519.198895,13156.920371,1.00,76.00,286.00,1209.00,138789.00,7.416351,65.578591
odds_box,181.0,189.061050,668.443559,0.05,3.95,16.65,71.38,6939.45,7.093851,60.980879
est_value_usd,194.0,287.324742,815.751158,0.50,16.00,100.00,150.00,5000.00,4.414497,18.452395
packs_per_box,194.0,18.206186,4.289856,8.00,20.00,20.00,20.00,20.00,-1.966065,1.865413
retail_price_usd,194.0,667.651495,336.901344,259.95,379.99,549.99,1099.95,1100.00,0.373870,-1.595295


## Stage 09 — features  →  `homework/homework09/`

`src/features.py`: `expected_hits_per_box` (the EV weight), `log_odds_pack` (linear rarity
scale), and a one-hot of `tier_group`. Feature definitions in the README.

In [16]:
feat = features.encode_tier_group(
    features.add_log_odds(
        features.add_expected_hits_per_box(df)
    )
)
feat.filter(regex="expected_hits_per_box|log_odds_pack|is_chase|^tg_").describe().T

,count,mean,std,min,25%,50%,75%,max
expected_hits_per_box,181.0,0.609062,2.280459,0.000144,0.014011,0.060060,0.253165,20.000000
log_odds_pack,181.0,2.479064,0.988136,0.000000,1.880814,2.456366,3.082426,5.142355
tg_auto,194.0,0.242268,0.429564,0.000000,0.000000,0.000000,0.000000,1.000000
tg_base,194.0,0.041237,0.199353,0.000000,0.000000,0.000000,0.000000,1.000000
tg_insert,194.0,0.144330,0.352333,0.000000,0.000000,0.000000,0.000000,1.000000
tg_parallel,194.0,0.556701,0.498060,0.000000,0.000000,1.000000,1.000000,1.000000
tg_variation,194.0,0.015464,0.123708,0.000000,0.000000,0.000000,0.000000,1.000000


## Stage 10a — linear regression  →  `notebooks/modeling-linear-regression.ipynb`

Regression track: predict tier value `log10(est_value_usd)` from `log_odds_pack` + the
`is_*` flags + a one-hot of `tier_group`. This is the model that would replace the crude
`est_value_usd` ladder in `src/ev.py` once real comps exist.

In [17]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

m10a = features.add_log_odds(df)
m10a["log_value"] = np.log10(m10a["est_value_usd"])
for b in ["is_numbered", "is_autograph", "is_ssp"]:
    m10a[b] = m10a[b].astype(int)
for g in ["parallel", "insert", "auto", "variation"]:
    m10a[f"tg_{g}"] = (m10a["tier_group"] == g).astype(int)

f10a = ["log_odds_pack", "is_numbered", "is_autograph", "is_ssp",
        "tg_parallel", "tg_insert", "tg_auto", "tg_variation"]
m10a = m10a.dropna(subset=f10a + ["log_value"]).reset_index(drop=True)

Xtr, Xte, ytr, yte = train_test_split(m10a[f10a], m10a["log_value"],
                                      test_size=0.2, random_state=7)
reg10a = LinearRegression().fit(Xtr, ytr)
p = reg10a.predict(Xte)
print(f"Stage 10a regression  R2={r2_score(yte, p):.3f}  "
      f"RMSE={np.sqrt(((yte - p) ** 2).mean()):.3f} (log10 USD)  "
      f"[label = placeholder ladder; see hw10a]")

Stage 10a regression  R2=0.896  RMSE=0.276 (log10 USD)  [label = placeholder ladder; see hw10a]


### Stage 10a via `src/model.py` — parity check (Stage 13)

The cell above is the original inline fit. `src/model.py` moves that exact logic into
`build_value_frame` + `evaluate_value_model` so `app.py` can reuse it. Re-run it here and
assert the holdout R² / RMSE are unchanged — that is how we know the refactor is safe.

In [18]:
# Stage 13 refactor check: src/model.py must reproduce the inline Stage 10a fit above.
from src import model as prod_model

_m = prod_model.evaluate_value_model(df, random_state=7)
print(f"src.model.evaluate_value_model   R2={_m['r2']:.3f}  RMSE={_m['rmse']:.3f}  "
      f"(n_train={_m['n_train']}, n_test={_m['n_test']})")

assert np.isclose(_m["r2"], r2_score(yte, p)),                  "R2 drifted from the inline cell"
assert np.isclose(_m["rmse"], np.sqrt(((yte - p) ** 2).mean())), "RMSE drifted from the inline cell"
print("parity OK -- the src/ refactor did not change the model before it goes into app.py")

src.model.evaluate_value_model   R2=0.896  RMSE=0.276  (n_train=144, n_test=37)
parity OK -- the src/ refactor did not change the model before it goes into app.py


## Stage 10b — classification  →  `notebooks/modeling-time-series-and-classification.ipynb`

No time series in this dataset, so the modeling track is classification: predict
`is_high_value` (`est_value_usd >= 100`) from tier-intrinsic features plus rarity-ladder
lag/rolling features (`src/features.py::add_ladder_features`), in a `StandardScaler ->
LogisticRegression` pipeline.

In [19]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

m10b = df.copy()
m10b["log_odds_pack"] = np.log10(m10b["odds_pack"])
m10b["expected_hits_per_box"] = m10b["packs_per_box"] / m10b["odds_pack"]
for b in ["is_numbered", "is_autograph", "is_ssp"]:
    m10b[b] = m10b[b].astype(int)
m10b["is_high_value"] = (m10b["est_value_usd"] >= 100).astype(int)
m10b = features.add_ladder_features(
    m10b.dropna(subset=["log_odds_pack", "expected_hits_per_box"]).reset_index(drop=True)
)

f10b = ["log_odds_pack", "is_numbered", "is_autograph", "is_ssp",
        "rarity_rank", "log_odds_gap_prev", "cum_exp_hits"]
Xa, Xb, ya, yb = train_test_split(m10b[f10b], m10b["is_high_value"], test_size=0.2,
                                  random_state=7, stratify=m10b["is_high_value"])
clf10b = Pipeline([("scaler", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=1000))]).fit(Xa, ya)
print("Stage 10b classifier  ROC-AUC =",
      round(roc_auc_score(yb, clf10b.predict_proba(Xb)[:, 1]), 3),
      "(label is the placeholder ladder -> near-deterministic; see hw10b)")

Stage 10b classifier  ROC-AUC = 0.985 (label is the placeholder ladder -> near-deterministic; see hw10b)


## Stage 11 — evaluation & risk  →  `homework/homework11/` · `data/processed/scenario_results.csv`

`src/evaluation.py`: scenario sensitivity (mean / median impute / drop the 13
unpublished-odds tiers) and a bootstrap CI on the value line's MAE. Full CIs, subgroup
residuals, and the stakeholder summary are in `homework/homework11/`.

In [20]:
x11 = np.log10(df["odds_pack"].values)
y11 = np.log10(df["est_value_usd"].values)
sens = evaluation.scenario_table(x11, y11, {"mean_impute": "mean",
                                            "median_impute": "median",
                                            "drop_missing": "drop"})
b0, b1 = evaluation.ols(evaluation.mean_impute(x11), y11)
boot = evaluation.bootstrap_metric(
    y11, evaluation.predict(evaluation.mean_impute(x11), b0, b1), evaluation.mae, seed=111)
print(sens.round(4).to_string(index=False))
print(f"\nbaseline MAE {boot['mean']:.3f}  95% CI [{boot['lo']:.3f}, {boot['hi']:.3f}] (log10 USD)")
print("-> slope stable across imputation scenarios; CIs / subgroup residuals / summary in hw11")

     scenario   n  slope  intercept    mae
  mean_impute 194 0.6688     0.0949 0.3190
median_impute 194 0.6700     0.0929 0.3187
 drop_missing 181 0.6688     0.1475 0.2540

baseline MAE 0.321  95% CI [0.270, 0.380] (log10 USD)
-> slope stable across imputation scenarios; CIs / subgroup residuals / summary in hw11


## Stage 12 — reporting & delivery  →  `reports/stakeholder_report.md`

`src/ev.py::ev_table()` ranks every box config by `EV/$` (the buy/pass signal) and writes
`data/processed/ev_report.csv`. The stakeholder-ready **written report** —
problem & method, results with charts, an **Assumptions & Risks** table with a
0.5x / 2x card-value sensitivity scenario, and decision implications — is
`reports/stakeholder_report.md` (`.pdf` alongside); charts in `reports/images/`.

In [21]:
report = ev.ev_table()
(project_root / "data" / "processed").mkdir(parents=True, exist_ok=True)
report.to_csv(project_root / "data" / "processed" / "ev_report.csv", index=False)

buys = report[report["ev_per_price"] >= 1]
print(f"{len(report)} box configs | {len(buys)} positive-EV (EV/$ >= 1):")
print(buys[["product", "price", "ev", "ev_per_price"]].to_string(index=False))
print("\nfull table -> data/processed/ev_report.csv")
print("stakeholder report -> reports/stakeholder_report.md (+ .pdf) ; charts -> reports/images/")

29 box configs | 3 positive-EV (EV/$ >= 1):
                        product  price     ev  ev_per_price
2025-26 Bowman Basketball Jumbo 599.99 756.80         1.261
  2025-26 Topps NBA Hoops Hobby 259.95 290.24         1.117
  2025-26 Topps NBA Hoops Jumbo 519.99 539.60         1.038

full table -> data/processed/ev_report.csv
stakeholder report -> reports/stakeholder_report.md (+ .pdf) ; charts -> reports/images/


## Stage 13 — productization  →  `src/model.py`, `src/plotting.py`, `app.py`

What this stage built, run here so the notebook proves it end to end:

1. **`src/model.py`** — the Stage 10a value model as reusable functions (`build_value_frame`,
   `train_value_model`, `save_model` / `load_model` / `get_model`, `predict_value` /
   `predict_from_odds`). `get_model()` loads `model/model.pkl` if it exists, else trains and
   saves it.
2. **`src/plotting.py`** — `ev_vs_price_figure()` for the `/plot` route.
3. **`app.py`** — Flask API on `http://127.0.0.1:5001` (`/predict`, `/predict/<odds>`,
   `/predict/<odds>/<tier_group>`, `/run_full_analysis`, `/plot`), model loaded once at
   startup, invalid input → HTTP 400 JSON.

Full write-up and setup: `README.md`; homework version: `homework13_productization_submission.ipynb`.

In [24]:
from src import model as prod_model, plotting as prod_plot

# save (overrides any existing) then reload -- the two options the deliverable asks for
_served = prod_model.train_value_model(df)
prod_model.save_model(_served)                     # -> model/model.pkl
reloaded = prod_model.get_model()                  # loads the file we just wrote
print("model/model.pkl written;", type(reloaded).__name__, "reloaded from disk")

# single-tier prediction through the exact code path app.py serves
for odds, tg in [(3, "parallel"), (500, "auto"), (30000, "auto")]:
    r = prod_model.predict_from_odds(reloaded, odds_pack=odds, tier_group=tg)
    print(f"  1:{odds:>6} {tg:<9} -> est_value_usd ~ ${r['est_value_usd']:,.0f}")

fig = prod_plot.ev_vs_price_figure(report)
fig.savefig(project_root / "reports" / "images" / "ev_vs_price_api.png",
            dpi=110, bbox_inches="tight")
print("chart -> reports/images/ev_vs_price_api.png  (served live at GET /plot)")

model/model.pkl written; LinearRegression reloaded from disk
  1:     3 parallel  -> est_value_usd ~ $2
  1:   500 auto      -> est_value_usd ~ $56
  1: 30000 auto      -> est_value_usd ~ $714
chart -> reports/images/ev_vs_price_api.png  (served live at GET /plot)


## Result

Ran top to bottom without errors — `src/cleaning.py`, `src/outliers.py`, `src/eda.py`,
`src/features.py` (Stages 09 + 10a/10b), `src/evaluation.py` (Stage 11), `src/ev.py`
(Stage 12 reporting), and `src/model.py` + `src/plotting.py` (Stage 13 productization) all
still work against the current project dataset. The Stage 10a parity check passed, so
`model/model.pkl` and `app.py` serve the same model the notebook fits. Per-stage write-ups: `notebooks/sensitivity_outliers.ipynb`, `notebooks/eda.ipynb`,
`homework/homework09/`, `notebooks/modeling-linear-regression.ipynb`,
`notebooks/modeling-time-series-and-classification.ipynb`, `homework/homework11/`,
`reports/stakeholder_report.md`, `README.md`, `homework/homework13/`, and `docs/`.